In [1]:
import numpy as np
import pandas as pd
import requests

In [2]:
#Importing countries through api
url="https://api.worldbank.org/countries?format=json&per_page=300"
response=requests.get(url)
response.status_code

200

In [3]:
data=response.json()
print(data[0])

{'page': 1, 'pages': 1, 'per_page': '300', 'total': 295}


In [4]:
countries=data[1]
countries=pd.DataFrame(countries)

In [5]:
countries

,id,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude
0,ABW,AW,Aruba,"{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Lati...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'HIC', 'iso2code': 'XD', 'value': 'High...","{'id': 'LNX', 'iso2code': 'XX', 'value': 'Not ...",Oranjestad,-70.0167,12.5167
1,AFE,ZH,Africa Eastern and Southern,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
2,AFG,AF,Afghanistan,"{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'MNA', 'iso2code': 'XQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","{'id': 'IDX', 'iso2code': 'XI', 'value': 'IDA'}",Kabul,69.1761,34.5228
3,AFR,A9,Africa,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
4,AFW,ZI,Africa Western and Central,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
...,...,...,...,...,...,...,...,...,...,...
290,XZN,A5,Sub-Saharan Africa excluding South Africa and ...,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
291,YEM,YE,"Yemen, Rep.","{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'MNA', 'iso2code': 'XQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","{'id': 'IDX', 'iso2code': 'XI', 'value': 'IDA'}",Sana'a,44.2075,15.352
292,ZAF,ZA,South Africa,"{'id': 'SSF', 'iso2code': 'ZG', 'value': 'Sub-...","{'id': 'SSA', 'iso2code': 'ZF', 'value': 'Sub-...","{'id': 'UMC', 'iso2code': 'XT', 'value': 'Uppe...","{'id': 'IBD', 'iso2code': 'XF', 'value': 'IBRD'}",Pretoria,28.1871,-25.746
293,ZMB,ZM,Zambia,"{'id': 'SSF', 'iso2code': 'ZG', 'value': 'Sub-...","{'id': 'SSA', 'iso2code': 'ZF', 'value': 'Sub-...","{'id': 'LMC', 'iso2code': 'XN', 'value': 'Lowe...","{'id': 'IDX', 'iso2code': 'XI', 'value': 'IDA'}",Lusaka,28.2937,-15.3982


In [6]:
countries['region']=countries['region'].apply(lambda x:x['value'])
countries['incomeLevel']= countries['incomeLevel'].apply(lambda x:x['value'])
countries['lendingType']= countries['lendingType'].apply(lambda x:x['value'])

In [7]:
countries.rename(columns={'iso2Code':'country_id'},inplace=True)
countries.drop(columns=['adminregion','lendingType','capitalCity'],inplace=True)

In [8]:
countries

,id,country_id,name,region,incomeLevel,longitude,latitude
0,ABW,AW,Aruba,Latin America & Caribbean,High income,-70.0167,12.5167
1,AFE,ZH,Africa Eastern and Southern,Aggregates,Aggregates,,
2,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,69.1761,34.5228
3,AFR,A9,Africa,Aggregates,Aggregates,,
4,AFW,ZI,Africa Western and Central,Aggregates,Aggregates,,
...,...,...,...,...,...,...,...
290,XZN,A5,Sub-Saharan Africa excluding South Africa and ...,Aggregates,Aggregates,,
291,YEM,YE,"Yemen, Rep.","Middle East, North Africa, Afghanistan & Pakistan",Low income,44.2075,15.352
292,ZAF,ZA,South Africa,Sub-Saharan Africa,Upper middle income,28.1871,-25.746
293,ZMB,ZM,Zambia,Sub-Saharan Africa,Lower middle income,28.2937,-15.3982


In [9]:
#code for indicators
ind_url='https://api.worldbank.org/v2/indicators?format=json'
response=requests.get(ind_url)
response.status_code

200

In [10]:
indicators_data=response.json()
indicators_data[0]

{'page': 1, 'pages': 591, 'per_page': '50', 'total': 29544}

In [11]:
all_dfs = []

for i in range(1, 592):
    url = f"https://api.worldbank.org/v2/indicators?format=json&per_page=500&page={i}"

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        if len(data) < 2 or not data[1]:
            print(f"No data at page {i}")

        df = pd.DataFrame([
            {"id": item["id"], "name": item["name"]}
            for item in data[1]
        ])

        all_dfs.append(df)
        print(f"Page {i}: {len(df)} indicators collected")

    else:
        print(f"Failed to fetch page {i}, status code {response.status_code}")



Page 1: 500 indicators collected
Page 2: 500 indicators collected
Page 3: 500 indicators collected
Page 4: 500 indicators collected
Page 5: 500 indicators collected
Page 6: 500 indicators collected
Page 7: 500 indicators collected
Page 8: 500 indicators collected
Page 9: 500 indicators collected
Page 10: 500 indicators collected
Page 11: 500 indicators collected
Page 12: 500 indicators collected
Page 13: 500 indicators collected
Page 14: 500 indicators collected
Page 15: 500 indicators collected
Page 16: 500 indicators collected
Page 17: 500 indicators collected
Page 18: 500 indicators collected
Page 19: 500 indicators collected
Page 20: 500 indicators collected
Page 21: 500 indicators collected
Page 22: 500 indicators collected
Page 23: 500 indicators collected
Page 24: 500 indicators collected
Page 25: 500 indicators collected
Page 26: 500 indicators collected
Page 27: 500 indicators collected
Page 28: 500 indicators collected
Page 29: 500 indicators collected
Page 30: 500 indicators

In [12]:
final_df=pd.concat(all_dfs,ignore_index=True)
final_df.to_csv('final_df.csv')

In [13]:
# extract value for various indicators for each country
indicator_groups = {
"economic_activity_growth": [
"NY.GDP.MKTP.KD.ZG", # GDP growth (annual %)
"NY.GDP.PCAP.CD" # GDP per capita (current US$)
],
"labour_market_indicators": [
"SL.UEM.TOTL.ZS", # Unemployment total
"SL.UEM.1524.ZS", # Unemployment youth total (ages 15–24)
"SL.TLF.TOTL.IN" # Labour force, total
],
"trade_globalization": [
"NE.EXP.GNFS.CD", # Exports of goods and services (current US$)
"NE.IMP.GNFS.CD" # Imports of goods and services (current US$)
],
"poverty_inequality": [
"SI.POV.NAHC", # Poverty headcount ratio at national poverty lines (% of population)
"SI.POV.GINI" # Gini index (measure of income inequality)
],
"environmental_indicators": [
"EG.FEC.RNEW.ZS", # Renewable energy consumption (% of total final energy consumption)
"AG.LND.FRST.ZS" # Forest area (% of land area)
],
"health_indicators": [
"SP.DYN.LE00.IN", # Life expectancy at birth
"SP.DYN.IMRT.IN", # Infant mortality rate
"SH.H2O.BASW.ZS", # Access to at least basic water services (% of population)
"SH.XPD.CHEX.GD.ZS", # Current health expenditure (% of GDP)
"SH.IMM.IDPT", # Immunization, DPT (% of children ages 12–23 months)
"SH.IMM.MEAS", # Immunization, measles (% of children ages 12–23 months)
"SH.MMR.RISK.ZS", # Risk of maternal death
"SH.DTH.COMM.ZS", # Deaths from communicable diseases (% of total)
"SH.TBS.INCD", # Tuberculosis incidence (per 100,000 people)
"SH.STA.BRTC.ZS", # Births attended by skilled health staff (%)
"SH.STA.MMRT", # Maternal mortality ratio (modeled estimate, per 100,000 live births)
"SP.POP.65UP.TO.ZS", # Population ages 65 and above (% of total population)
"SH.HIV.INCD.ZS" # HIV incidence rate (per 1,000 uninfected population ages 15–49)
],
"technology_indicators": [
"IT.NET.USER.ZS", # Individuals using the Internet (% of population)
"IT.CEL.SETS.P2" # Mobile cellular subscriptions (per 100 people)
]}

In [14]:
for i,j in indicator_groups.items():
  print(j)


['NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD']
['SL.UEM.TOTL.ZS', 'SL.UEM.1524.ZS', 'SL.TLF.TOTL.IN']
['NE.EXP.GNFS.CD', 'NE.IMP.GNFS.CD']
['SI.POV.NAHC', 'SI.POV.GINI']
['EG.FEC.RNEW.ZS', 'AG.LND.FRST.ZS']
['SP.DYN.LE00.IN', 'SP.DYN.IMRT.IN', 'SH.H2O.BASW.ZS', 'SH.XPD.CHEX.GD.ZS', 'SH.IMM.IDPT', 'SH.IMM.MEAS', 'SH.MMR.RISK.ZS', 'SH.DTH.COMM.ZS', 'SH.TBS.INCD', 'SH.STA.BRTC.ZS', 'SH.STA.MMRT', 'SP.POP.65UP.TO.ZS', 'SH.HIV.INCD.ZS']
['IT.NET.USER.ZS', 'IT.CEL.SETS.P2']


In [15]:
import time

session = requests.Session()
base_url="https://api.worldbank.org/countries/all/indicators/{}?format=json&per_page=1000&page={}"
category_dataframes = {}

for category, indicators in indicator_groups.items():
    print(f"Fetching {category}")

    rows = []

    for indicator in indicators:
        page = 1

        while True:
            url = base_url.format(indicator, page)

            response = session.get(url)

            if response.status_code != 200:
                print(f"Failed: {indicator}")
                break

            data = response.json()

            if len(data) < 2 or not data[1]:
                break

            total_pages = data[0]["pages"]

            for r in data[1]:
                year = int(r["date"])

                if year > 2015:
                    rows.append({
                        "country_id": r["country"]["id"],
                        "country_name": r["country"]["value"],
                        "indicator_id": r["indicator"]["id"],
                        "indicator_name": r["indicator"]["value"],
                        "year": year,
                        "value": r["value"]
                    })

            if page >= total_pages:
                break

            page += 1
            time.sleep(0.2)

    category_dataframes[category] = pd.DataFrame(rows)

    print(f"{category}: {len(rows)} rows")

Fetching economic_activity_growth
economic_activity_growth: 5300 rows
Fetching labour_market_indicators
labour_market_indicators: 7950 rows
Fetching trade_globalization
trade_globalization: 5300 rows
Fetching poverty_inequality
poverty_inequality: 5300 rows
Fetching environmental_indicators
environmental_indicators: 5300 rows
Fetching health_indicators
health_indicators: 34450 rows
Fetching technology_indicators
technology_indicators: 5300 rows


In [16]:
economic_activity= category_dataframes.get("economic_activity_growth", pd.DataFrame())
labour_market_jobs= category_dataframes.get("labour_market_indicators", pd.DataFrame())
trade_globalization= category_dataframes.get("trade_globalization", pd.DataFrame())
poverty_inequality= category_dataframes.get("poverty_inequality", pd.DataFrame())
environmental_indicators= category_dataframes.get("environmental_indicators", pd.DataFrame())
health_indicators= category_dataframes.get("health_indicators", pd.DataFrame())
technology_indicators= category_dataframes.get("technology_indicators", pd.DataFrame())

In [17]:
economic= pd.merge(economic_activity, countries, on ="country_id", how="inner")
labour_market = pd.merge(labour_market_jobs, countries, on ="country_id", how="inner")
trade= pd.merge(trade_globalization, countries, on ="country_id", how="inner")
poverty= pd.merge(poverty_inequality, countries, on ="country_id", how="inner")
environment= pd.merge(environmental_indicators, countries, on ="country_id", how="inner")
health=pd.merge(health_indicators, countries, on ="country_id", how="inner")
technology=pd.merge(technology_indicators, countries, on ="country_id", how="inner")

In [18]:
economic.drop(columns=["indicator_id","name","id"], inplace=True)
trade.drop(columns=["indicator_id","name","id"], inplace=True)
labour_market.drop(columns=["indicator_id","name","id"], inplace=True)
poverty.drop(columns=["indicator_id","name","id"], inplace=True)
environment.drop(columns=["indicator_id","name","id"], inplace=True)
health.drop(columns=["indicator_id","name","id"], inplace=True)
technology.drop(columns=["indicator_id","name","id"], inplace=True)

In [19]:
economic

,country_id,country_name,indicator_name,year,value,region,incomeLevel,longitude,latitude
0,ZH,Africa Eastern and Southern,GDP growth (annual %),2025,3.743016,Aggregates,Aggregates,,
1,ZH,Africa Eastern and Southern,GDP growth (annual %),2024,2.787931,Aggregates,Aggregates,,
2,ZH,Africa Eastern and Southern,GDP growth (annual %),2023,1.941382,Aggregates,Aggregates,,
3,ZH,Africa Eastern and Southern,GDP growth (annual %),2022,3.668029,Aggregates,Aggregates,,
4,ZH,Africa Eastern and Southern,GDP growth (annual %),2021,4.452643,Aggregates,Aggregates,,
...,...,...,...,...,...,...,...,...,...
5295,ZW,Zimbabwe,GDP per capita (current US$),2020,2059.637040,Sub-Saharan Africa,Lower middle income,31.0672,-17.8312
5296,ZW,Zimbabwe,GDP per capita (current US$),2019,2184.521554,Sub-Saharan Africa,Lower middle income,31.0672,-17.8312
5297,ZW,Zimbabwe,GDP per capita (current US$),2018,2270.895319,Sub-Saharan Africa,Lower middle income,31.0672,-17.8312
5298,ZW,Zimbabwe,GDP per capita (current US$),2017,3445.449410,Sub-Saharan Africa,Lower middle income,31.0672,-17.8312


In [20]:
economic.to_csv('economic.csv')
labour_market.to_csv('labour_market.csv')
trade.to_csv('trade.csv')
poverty.to_csv('poverty.csv')
environment.to_csv('environment.csv')
health.to_csv('health.csv')
technology.to_csv('technology.csv')

NameError: name 'technology' is not defined